# Task 2: Exploratory Data Analysis

Analyze patterns and relationships in Ethiopia's financial inclusion data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load enriched dataset
df = pd.read_csv('../data/enriched/ethiopia_fi_unified_data_enriched.csv')
df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')

print("Summary by record_type:")
print(df['record_type'].value_counts())

obs = df[df['record_type'] == 'observation']
print("\nSummary by pillar (observations only):")
print(obs['pillar'].value_counts(dropna=False))

print("\nSource type distribution:")
print(df['source_name'].value_counts().head(10))

print("\nConfidence distribution:")
print(df['confidence'].value_counts(dropna=False))

In [ ]:
# Temporal coverage visualization
obs_pivot = obs.pivot(index='observation_date', columns='indicator_code', values='value_numeric')
obs_pivot.plot(marker='o', figsize=(12,6))
plt.title('Temporal Coverage of Indicators')
plt.ylabel('Value')
plt.grid(True)
plt.tight_layout()
plt.show()

## Access Analysis

In [ ]:
# Account ownership trend
acc = obs[obs['indicator_code'] == 'ACC_OWNERSHIP'].copy()
acc = acc.sort_values('observation_date')

plt.figure(figsize=(10,6))
plt.plot(acc['observation_date'], acc['value_numeric'], marker='o', linestyle='-', linewidth=2)
plt.title('Ethiopia Account Ownership (2011-2024)')
plt.ylabel('Percentage of adults')
plt.grid(True)
plt.show()

# Growth rates
acc['growth_pp'] = acc['value_numeric'].diff()
acc['growth_pct'] = acc['value_numeric'].pct_change() * 100
print(acc[['observation_date', 'value_numeric', 'growth_pp', 'growth_pct']])

In [ ]:
# Gender/urban disaggregation if available
gender_cols = [c for c in obs['indicator_code'].unique() if 'ACC_OWNERSHIP_' in c and c != 'ACC_OWNERSHIP']
if gender_cols:
    gender = obs[obs['indicator_code'].isin(gender_cols)]
    pivot_gender = gender.pivot(index='observation_date', columns='indicator_code', values='value_numeric')
    pivot_gender.plot(marker='o')
    plt.title('Account Ownership by Gender')
    plt.ylabel('Percentage')
    plt.grid(True)
    plt.show()
else:
    print("No gender-disaggregated data available.")

## Usage (Digital Payments) Analysis

In [ ]:
# Mobile money account penetration
mm = obs[obs['indicator_code'] == 'ACC_MM_ACCOUNT'].sort_values('observation_date')
if not mm.empty:
    plt.figure()
    plt.plot(mm['observation_date'], mm['value_numeric'], marker='s', color='green')
    plt.title('Mobile Money Account Penetration')
    plt.ylabel('Percentage of adults')
    plt.grid(True)
    plt.show()

# Digital payment adoption
dig = obs[obs['indicator_code'] == 'USG_DIGITAL_PAYMENT'].sort_values('observation_date')
if not dig.empty:
    plt.figure()
    plt.plot(dig['observation_date'], dig['value_numeric'], marker='^', color='orange')
    plt.title('Digital Payment Adoption')
    plt.ylabel('Percentage of adults')
    plt.grid(True)
    plt.show()

## Infrastructure and Enablers

In [ ]:
infra_codes = ['INF_4G_COVERAGE', 'INF_MOBILE_PEN', 'INF_ATM_DENSITY', 'INF_BANK_BRANCHES']
infra = obs[obs['indicator_code'].isin(infra_codes)]
if not infra.empty:
    infra_pivot = infra.pivot(index='observation_date', columns='indicator_code', values='value_numeric')
    infra_pivot.plot(subplots=True, layout=(2,2), figsize=(12,8), marker='o')
    plt.suptitle('Infrastructure Indicators', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Infrastructure data not found in the dataset.")

## Event Timeline and Visual Analysis

In [ ]:
events = df[df['record_type'] == 'event']
acc_plot = obs[obs['indicator_code'] == 'ACC_OWNERSHIP'].sort_values('observation_date')

plt.figure(figsize=(14,7))
plt.plot(acc_plot['observation_date'], acc_plot['value_numeric'], marker='o', label='Account Ownership')
for idx, row in events.iterrows():
    plt.axvline(x=row['observation_date'], color='red', linestyle='--', alpha=0.5, linewidth=1)
    plt.text(row['observation_date'], acc_plot['value_numeric'].max()*0.9, 
             row['indicator'][:20], rotation=45, fontsize=8)
plt.title('Account Ownership with Events')
plt.ylabel('Percentage')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
# Create wide format and correlation matrix
obs_wide = obs.pivot(index='observation_date', columns='indicator_code', values='value_numeric')
corr = obs_wide.corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Indicators')
plt.show()

# Focus on target variables
targets = ['ACC_OWNERSHIP', 'USG_DIGITAL_PAYMENT']
if all(t in obs_wide.columns for t in targets):
    corr_target = obs_wide[targets].corrwith(obs_wide)
    print("Correlation with Access and Usage:")
    print(corr_target.sort_values(ascending=False))

## Key Insights Summary

Update the text below based on your actual observations from the data.

In [ ]:
insights = """
### Key Insights from EDA

1. Stagnating account ownership: Only +3pp from 2021 to 2024 despite Telebirr growth.
2. Mobile money accounts surged: From 4.7% to 9.45% (2021–2024).
3. Event correlation: Telebirr launch aligns with mobile money growth, but not account ownership.
4. Infrastructure enablers: 4G and mobile penetration correlate with digital payments.
5. Data gaps: Sparse annual data limits causal inference.
6. Potential leading indicator: Mobile money account growth may lead digital payment usage.
7. Interoperability policy: New NBE mandate could boost digital payments but effect uncertain.
"""
print(insights)